# Lab 2 — LoRA fine-tuning of OpenVLA on a SageMaker Training Job

| | Serverless SFT lab | This lab |
|---|---|---|
| API | `SFTTrainer` (managed recipe) | `ModelTrainer` + `scripts/` |
| Compute | chosen by the service | `ml.g6e.xlarge` (smoke) / `ml.g6e.12xlarge` (full), managed Spot |
| Container | managed | PyTorch 2.7 training DLC + `scripts/requirements.txt` |
| Distribution | managed | Accelerate DDP, one process per GPU |
| Tracking | MLflow App | same MLflow App (`MLFLOW_TRACKING_URI` = app ARN) |
| Output | model package + `hf_merged/` | `adapter/` + `hf_merged/`, registered as a model package |

**Training recipe** (`scripts/recipes/openvla_lora.yaml`, from `openvla/vla-scripts/finetune.py`):
LoRA r=32, alpha=16, dropout 0, `all-linear` targets (~111 M trainable params, 1.4 %), AdamW lr 5e-4
cosine, bf16, 4 samples/GPU, global batch 32 (gradient accumulation fills the gap on fewer GPUs).
The loss covers only the 7 action tokens + EOS.

**Prerequisites:** a Spot quota for the training instance type (`ml.g6e.xlarge for spot training job usage`
for the smoke run, `ml.g6e.12xlarge for spot training job usage` for the full run). With `USE_SPOT = False`
the on-demand `... for training job usage` quotas apply instead. `openvla/openvla-7b` is
public, so no Hugging Face token is needed.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE} -> {config.TRAIN['instance_type']}")
print(f"s3 root:  {s3_root}")

In [ ]:
from config import resolve_mlflow_app

mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess)

## Configure the job

* `OutputDataConfig(compression_type="NONE")` writes the merged 15 GB checkpoint uncompressed, so it
  can be served straight from S3 (same idea as `hf_merged/` in the serverless lab).
* `CheckpointConfig` syncs `/opt/ml/checkpoints` (LoRA adapter + optimizer state) to S3. With managed Spot,
  SageMaker restarts an interrupted job and the script resumes from the latest checkpoint. The job may sit in
  `Starting` while it waits for Spot capacity (up to `SPOT_MAX_WAIT_H`).

In [ ]:
import time

from sagemaker.core.training.configs import (CheckpointConfig, Compute, InputData, OutputDataConfig,
                                             SourceCode, StoppingCondition)
from sagemaker.train import ModelTrainer

run_name = f"{config.PROJECT}-{config.RUN_MODE}-{time.strftime('%Y%m%d-%H%M%S')}"
max_runtime = config.TRAIN["max_runtime_h"] * 3600

trainer = ModelTrainer(
    training_image=config.TRAINING_IMAGE.format(region=region),
    source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="run_finetuning.sh"),
    compute=Compute(instance_type=config.TRAIN["instance_type"], instance_count=1, volume_size_in_gb=200,
                    enable_managed_spot_training=config.USE_SPOT),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=max_runtime,
                                         max_wait_time_in_seconds=max(max_runtime, config.SPOT_MAX_WAIT_H * 3600) if config.USE_SPOT else None),
    output_data_config=OutputDataConfig(s3_output_path=f"{s3_root}/training-output", compression_type="NONE"),
    checkpoint_config=CheckpointConfig(s3_uri=f"{s3_root}/checkpoints/{run_name}", local_path="/opt/ml/checkpoints"),
    environment={
        "RECIPE": config.TRAIN["recipe"],
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": config.MLFLOW_EXPERIMENT,
        "NCCL_DEBUG": "WARN",
    },
    base_job_name=run_name[:48],
    role=role,
    sagemaker_session=sess,
)
print(f"run: {run_name}\nrecipe: {config.TRAIN['recipe']}\ndata: {dataset_s3_uri}")

In [ ]:
def wait_for_job(job_name, poll=60):
    """Prints status transitions and the latest [step ...] log line instead of streaming every log line
    (the full stream can exceed Jupyter's output rate limit)."""
    logs = boto3.client("logs", region_name=region)
    print(f"https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/jobs/{job_name}")
    last = None
    while True:
        d = sm_client.describe_training_job(TrainingJobName=job_name)
        step = ""
        try:
            events = logs.filter_log_events(logGroupName="/aws/sagemaker/TrainingJobs", logStreamNamePrefix=job_name,
                                            filterPattern='"[step"', startTime=int((time.time() - 900) * 1000))["events"]
            step = events[-1]["message"].strip()[:160] if events else ""
        except logs.exceptions.ResourceNotFoundException:
            pass
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']} {step}"
        if line != last:
            print(time.strftime("%H:%M:%S"), line, flush=True)
            last = line
        if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped"):
            if d["TrainingJobStatus"] != "Completed":
                raise RuntimeError(d.get("FailureReason", d["TrainingJobStatus"]))
            return d
        time.sleep(poll)

## Train

`wait_for_job` polls the job and prints its status plus the latest `[step N]` line every minute (the full log
stream is in CloudWatch, linked below). The `[step N] train/... val/...` lines are also in MLflow
(Studio > MLflow > experiment `openvla-lora-bridge`). The job ends by merging the adapter into the base
model on the CPU, which adds a few minutes after the last step.

In [ ]:
trainer.train(input_data_config=[InputData(channel_name="train", data_source=dataset_s3_uri)], wait=False)
job_name = trainer._latest_training_job.training_job_name
job = wait_for_job(job_name)

In [ ]:
model_prefix = job["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
merged_s3_uri = model_prefix + "hf_merged/"

bucket, key = merged_s3_uri[5:].split("/", 1)
merged_files = [o["Key"].rsplit("/", 1)[-1] for o in s3.list_objects_v2(Bucket=bucket, Prefix=key)["Contents"]]
assert "config.json" in merged_files and "modeling_prismatic.py" in merged_files, merged_files
print(f"job:    {job_name} ({job['TrainingJobStatus']}, {job['TrainingTimeInSeconds'] // 60} min training, "
      f"{job['BillableTimeInSeconds'] // 60} min billed)")
print(f"merged: {merged_s3_uri}")
print("\n".join(sorted(merged_files)))

## Register the model

The serving handler (`serving/inference.py`, plus `scripts/openvla_utils.py`) is stored next to the
weights under `hf_merged/code/`. The PyTorch inference container loads it from `/opt/ml/model/code`.
The package is registered as `PendingManualApproval`. Lab 3 approves it if it beats the base model.

In [ ]:
for local, name in [("serving/inference.py", "inference.py"), ("serving/requirements.txt", "requirements.txt"),
                    ("scripts/openvla_utils.py", "openvla_utils.py")]:
    s3.upload_file(local, bucket, f"{key}code/{name}")

try:
    sm_client.describe_model_package_group(ModelPackageGroupName=config.MODEL_PACKAGE_GROUP)
except sm_client.exceptions.ClientError:
    sm_client.create_model_package_group(ModelPackageGroupName=config.MODEL_PACKAGE_GROUP,
                                         ModelPackageGroupDescription="OpenVLA-7B LoRA, BridgeData V2 (pseudo-labelled)")

package = sm_client.create_model_package(
    ModelPackageGroupName=config.MODEL_PACKAGE_GROUP,
    ModelPackageDescription=f"{config.BASE_MODEL_ID} + LoRA ({config.RUN_MODE}) from {job_name}",
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"training_job": job_name, "run_mode": config.RUN_MODE,
                                "merged_s3_uri": merged_s3_uri},
    InferenceSpecification={
        "Containers": [{
            "Image": config.INFERENCE_IMAGE.format(region=region),
            "ModelDataSource": {"S3DataSource": {"S3Uri": merged_s3_uri, "S3DataType": "S3Prefix",
                                                 "CompressionType": "None"}},
            "Environment": {
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                "SAGEMAKER_MODEL_SERVER_WORKERS": "1",
                "SAGEMAKER_TS_RESPONSE_TIMEOUT": "900",
                "SAGEMAKER_MODEL_SERVER_TIMEOUT": "900",
            },
        }],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": config.ENDPOINT_INSTANCE_TYPES,
    },
)
print(f"model package: {package['ModelPackageArn']}")

Continue with **3-evaluation.ipynb**.